# 02 · The dataset: what was collected, how clean it is, and why it is enough

Everything here was recorded by my own collectors, which have run continuously since **21 March 2026**.
None of it can be downloaded after the fact. The exchange's API only keeps about two months of candles
and fills, and it does not keep sub-second order-book history at all.

In [ ]:
import sys, json, warnings
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from tabtrader import plots
plots.style()
pd.set_option("display.width", 140, "display.max_columns", 30)

In [ ]:
from tabtrader import eda
t = eda.eda_tables()
pd.DataFrame(eda.MILESTONES, columns=["since", "what is recorded"])

In [ ]:
v = eda.volume_totals(t)
pd.Series({
  "15-minute market tape (rows)": v["tape_15m_rows"],
  "hourly ladder tape (rows)": v["tape_1h_rows"],
  "L2 order-book snapshots": v["order_book_rows"],
  "per-venue BTC spot events": v["spot_venue_rows"],
  "derived indicator rows": v["indicator_rows"],
  "BRTI settlement-index prints": v["brti_rows"],
  "settled 15-minute markets (labels)": v["labels_15m"],
  "settled hourly ladder legs (labels)": v["labels_1h_all_legs"],
}).map("{:,}".format).to_frame("count")

### Coverage: rows per day

In [ ]:
d15, d1h = t["tape_15m_daily"], t["tape_1h_daily"]
fig, ax = plt.subplots(figsize=(9, 3.2))
ax.plot(d15.day, d15.rows / 1e3, color=plots.TABPFN, label="15-minute tape")
ax.plot(d1h.day, d1h.rows / 1e3, color=plots.FAST, label="hourly tape")
ax.set_ylabel("thousand rows / day"); ax.legend(); ax.set_title("Collection never stopped")
plt.tight_layout()

### Quality: the collector upgrade on 30 July

`quote_valid` marks a two-sided, uncrossed, non-sentinel quote. The v1 collector stored about 60%
valid rows, and 2–4% of its rows were *crossed* (ask < bid, an impossible book). The v2 collector
fixed both. Earlier research on this data found that crossed rows produce phantom "edges" in
backtests, so they are excluded everywhere here.

In [ ]:
w = eda.weekly_quality(t)
fig, (a, b) = plt.subplots(1, 2, figsize=(10, 3.2))
for mk, col in [("15-minute", plots.TABPFN), ("hourly", plots.FAST)]:
    g = w[w.market == mk]
    a.plot(g.week, g.valid_pct, color=col, label=mk); b.plot(g.week, g.crossed_pct, color=col, label=mk)
for ax in (a, b): ax.axvline(pd.Timestamp("2026-07-30"), color=plots.MUTED, ls="--", lw=1)
a.set_title("valid quotes, % of rows"); b.set_title("crossed books, % of rows"); a.legend()
plt.tight_layout()

### Every decision quote checked against the exchange

For every decision snapshot used in modelling, my recorded quote is compared with Kalshi's own
1-minute candle for that minute. A snapshot is **confirmed** if it lies inside the candle's range and
within 2 ¢ of the exchange's closing quote. The hourly tape before the v2 upgrade fails this check
most of the time, so it is excluded.

In [ ]:
pd.concat({k: eda.candle_agreement(k).set_index("month") for k in ["15m", "1h"]}, axis=1).round(1)

### Is the market's own price already a good forecast?

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9, 4))
for ax, k, name in [(axes[0], "15m", "15-minute"), (axes[1], "1h", "hourly")]:
    c = eda.market_calibration(k)
    c = c[c.n >= 50]
    ax.plot([0, 1], [0, 1], color=plots.MUTED, ls="--", lw=1)
    ax.errorbar(c.p, c.y, yerr=[c.y - c.lo, c.hi - c.y], fmt="o", color=plots.TABPFN, ms=5, capsize=2)
    ax.set_xlabel("market mid price"); ax.set_ylabel("settled YES rate"); ax.set_title(name)
plt.tight_layout()

Very nearly. Each 10¢ price bucket settles close to its price, with a slight lean at the extremes
(favourites win a little more often than their price says). So the market is a **strong baseline**, and
any model has to beat the price, not a coin flip. That is why every comparison in notebook 04 includes
the market itself.

In [ ]:
lab = t["labels_daily"]
lab.groupby("series").agg(markets=("markets", "sum"), yes=("yes", "sum")).assign(
    yes_rate=lambda d: (d.yes / d.markets).round(3))